In [14]:
# Load environment variables from .env file (optional but recommended for Langfuse setup)
import os
from pathlib import Path

# Check multiple locations for .env file
env_paths = [
    Path(".env"),  # Current working directory
    Path(__file__).parent.parent / ".env" if '__file__' in locals() else None,  # Notebook's parent directory
    Path("/Users/manasdubey/Desktop/rag-system-elastic/.env"),  # Project root (absolute)
]

env_file = None
for candidate in env_paths:
    if candidate and candidate.exists():
        env_file = candidate
        break

if env_file:
    from dotenv import load_dotenv
    load_dotenv(env_file)
    print(f"✓ Loaded .env file from: {env_file}")
else:
    print("ℹ️ No .env file found. Set LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY manually if needed.")
    print(f"   Checked: {[str(p) for p in env_paths if p]}")

# Check Langfuse setup
langfuse_ready = bool(os.getenv("LANGFUSE_PUBLIC_KEY")) and bool(os.getenv("LANGFUSE_SECRET_KEY"))
print(f"Langfuse tracing: {'✓ ENABLED' if langfuse_ready else '✗ DISABLED (set env vars to enable)'}")

✓ Loaded .env file from: /Users/manasdubey/Desktop/rag-system-elastic/.env
Langfuse tracing: ✓ ENABLED


# Reflective RAG Agent — Production (Ollama + Langfuse + Gradio)

**6-step reflective reasoning loop:**
1. **Plan:** Decompose question into sub-questions
2. **Retrieve & Read:** Search v1 API, read top chunks
3. **Check Sufficiency:** Validate evidence covers all sub-questions
4. **Draft:** Generate structured answer with Ollama LLM
5. **Reflect:** Validate claims have citations
6. **Terminate:** Return final answer + confidence + metadata

**Dependencies:** Managed via UV. Ensure `uv sync --extra notebook` is run.

## Prerequisites & Configuration

**Service Requirements:**
- RAG API running at `http://localhost:8000` (v1 pipeline)
- Optional: Ollama at `http://localhost:11434` for local generation
- Optional: Langfuse Cloud account for tracing (free tier available at https://cloud.langfuse.com)

**Environment variables (set via `.env` file or shell):**
```bash
# Required for Langfuse tracing
LANGFUSE_PUBLIC_KEY=pk_...
LANGFUSE_SECRET_KEY=sk_...
LANGFUSE_HOST=https://cloud.langfuse.com

# Optional: customize endpoints
API_BASE_URL=http://localhost:8000
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=llama3.1:8b
```

In [16]:
# === IMPORTS ===
from __future__ import annotations

import time
import uuid
import re
import requests
from contextlib import contextmanager
from typing import Any, Dict, List, Literal, TypedDict, Optional

from langgraph.graph import StateGraph, END
import gradio as gr

try:
    from langfuse import Langfuse
except Exception:
    Langfuse = None

In [32]:
# === CONFIGURATION ===

# Retrieval (v1 API)
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
SEARCH_MODE = os.getenv("SEARCH_MODE", "dense_bm25")
TOP_K = int(os.getenv("TOP_K", "5"))
MAX_TOOL_CALLS = int(os.getenv("MAX_TOOL_CALLS", "8"))
MAX_SUBQUESTIONS = int(os.getenv("MAX_SUBQUESTIONS", "4"))
TIMEOUT_SECONDS = int(os.getenv("TIMEOUT_SECONDS", "30"))

# Local LLM (Ollama)
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "tinyllama:latest")
OLLAMA_TIMEOUT_SECONDS = int(os.getenv("OLLAMA_TIMEOUT_SECONDS", "60"))
ENABLE_OLLAMA_GENERATION = os.getenv("ENABLE_OLLAMA_GENERATION", "true").lower() == "true"

# Tracing (Langfuse)
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY", "")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY", "")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
ENABLE_LANGFUSE = bool(LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY)

print("\n=== Configuration ===")
print(f"Retrieval: {API_BASE_URL} | mode={SEARCH_MODE} | top_k={TOP_K}")
print(f"Generation: {OLLAMA_MODEL if ENABLE_OLLAMA_GENERATION else 'DISABLED'}")
print(f"Tracing: {'✓ ENABLED' if ENABLE_LANGFUSE else '✗ DISABLED'}")
print()


=== Configuration ===
Retrieval: http://localhost:8000 | mode=dense_bm25 | top_k=5
Generation: tinyllama:latest
Tracing: ✓ ENABLED



In [18]:
# === AGENT STATE ===

class AgentState(TypedDict):
    question: str
    sub_questions: List[str]
    current_query: str
    scratchpad: List[str]
    known_facts: List[str]
    open_gaps: List[str]
    retrieved_chunks: Dict[str, Dict[str, Any]]
    read_chunk_ids: List[str]
    evidence_by_question: Dict[str, List[str]]
    tool_calls_used: int
    enough_evidence: bool
    draft: str
    claims: List[str]
    unsupported_claims: List[str]
    need_more_retrieval: bool
    final_answer: str
    confidence: Literal["high", "medium", "low"]
    status: Literal["ok", "insufficient_evidence"]
    refused_reason: str
    request_id: str
    warnings: List[str]
    _trace: Any


def init_state(question: str) -> AgentState:
    return {
        "question": question,
        "sub_questions": [],
        "current_query": question,
        "scratchpad": [],
        "known_facts": [],
        "open_gaps": [],
        "retrieved_chunks": {},
        "read_chunk_ids": [],
        "evidence_by_question": {},
        "tool_calls_used": 0,
        "enough_evidence": False,
        "draft": "",
        "claims": [],
        "unsupported_claims": [],
        "need_more_retrieval": False,
        "final_answer": "",
        "confidence": "low",
        "status": "insufficient_evidence",
        "refused_reason": "",
        "request_id": "",
        "warnings": [],
        "_trace": None,
    }

In [19]:
# === HELPER FUNCTIONS ===

def budget_remaining(state: AgentState) -> int:
    return max(0, MAX_TOOL_CALLS - state["tool_calls_used"])


def search_tool(query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    payload = {
        "question": query,
        "top_k": top_k,
        "search_mode": SEARCH_MODE,
        "generate_answer": False,
        "pipeline_version": "v1",
    }
    resp = requests.post(f"{API_BASE_URL}/query", json=payload, timeout=TIMEOUT_SECONDS)
    resp.raise_for_status()
    data = resp.json()
    return data.get("results", [])


def read_chunk_tool(state: AgentState, chunk_id: str) -> Dict[str, Any] | None:
    return state["retrieved_chunks"].get(chunk_id)


def decompose_question(question: str, max_parts: int = MAX_SUBQUESTIONS) -> List[str]:
    rough = re.split(r"\?|\band\b|\bthen\b|\balso\b", question, flags=re.IGNORECASE)
    cleaned = [p.strip(" .") for p in rough if p.strip()]
    uniq: List[str] = []
    for part in cleaned:
        if part not in uniq:
            uniq.append(part)
    return uniq[:max_parts] if uniq else [question]

In [20]:
# === BASE NODE IMPLEMENTATIONS ===

def plan_node(state: AgentState) -> AgentState:
    subqs = decompose_question(state["question"])
    state["sub_questions"] = subqs
    state["open_gaps"] = subqs.copy()
    state["scratchpad"].append(f"Plan: decomposed into {len(subqs)} sub-questions")
    return state


def retrieve_read_node(state: AgentState) -> AgentState:
    if budget_remaining(state) <= 0:
        state["scratchpad"].append("Budget exhausted before retrieval")
        return state

    query = state["current_query"]
    try:
        results = search_tool(query, top_k=TOP_K)
    except Exception as e:
        state["scratchpad"].append(f"Search failed for '{query}': {e}")
        return state

    state["tool_calls_used"] += 1
    state["scratchpad"].append(f"search('{query}') -> {len(results)} results")

    if not results:
        state["open_gaps"] = list(dict.fromkeys(state["open_gaps"] + [query]))
        return state

    to_read = []
    for item in results[:2]:
        cid = item.get("chunk_id")
        if cid:
            to_read.append(cid)
        state["retrieved_chunks"][cid] = item

    for cid in to_read:
        if budget_remaining(state) <= 0:
            break
        chunk = read_chunk_tool(state, cid)
        if not chunk:
            continue
        state["tool_calls_used"] += 1
        if cid not in state["read_chunk_ids"]:
            state["read_chunk_ids"].append(cid)

        linked_subq = state["open_gaps"][0] if state["open_gaps"] else state["question"]
        state["evidence_by_question"].setdefault(linked_subq, [])
        if cid not in state["evidence_by_question"][linked_subq]:
            state["evidence_by_question"][linked_subq].append(cid)

        text = (chunk.get("content") or "").strip()
        if text:
            state["known_facts"].append(text[:280])

    if state["open_gaps"]:
        state["current_query"] = state["open_gaps"][0]

    return state


def check_sufficiency_node(state: AgentState) -> AgentState:
    unresolved: List[str] = []
    for sq in state["sub_questions"]:
        cits = state["evidence_by_question"].get(sq, [])
        if not cits:
            unresolved.append(sq)

    state["open_gaps"] = unresolved
    state["enough_evidence"] = len(unresolved) == 0 and len(state["read_chunk_ids"]) > 0
    if unresolved:
        state["scratchpad"].append(f"Sufficiency: missing evidence for {len(unresolved)} sub-questions")
        state["current_query"] = unresolved[0]
    else:
        state["scratchpad"].append("Sufficiency: enough evidence gathered")
    return state


def draft_node(state: AgentState) -> AgentState:
    if not state["read_chunk_ids"]:
        state["draft"] = ""
        return state

    lines: List[str] = []
    for sq in state["sub_questions"]:
        cits = state["evidence_by_question"].get(sq, [])
        if not cits:
            continue
        cid = cits[0]
        chunk = state["retrieved_chunks"].get(cid, {})
        snippet = (chunk.get("content") or "").strip().replace("\n", " ")
        snippet = (snippet[:220] + "...") if len(snippet) > 220 else snippet
        lines.append(f"- {sq}: {snippet} [{cid}]")

    state["draft"] = "\n".join(lines)
    state["claims"] = [ln for ln in lines if ln.strip()]
    state["scratchpad"].append(f"Drafted {len(state['claims'])} claim(s)")
    return state


def reflect_node(state: AgentState) -> AgentState:
    unsupported: List[str] = []
    for claim in state["claims"]:
        has_citation = bool(re.search(r"\[[^\]]+\]$", claim.strip()))
        if not has_citation:
            unsupported.append(claim)
            continue
        cid = claim.rsplit("[", 1)[-1].rstrip("]")
        if cid not in state["read_chunk_ids"]:
            unsupported.append(claim)

    state["unsupported_claims"] = unsupported

    if unsupported and budget_remaining(state) > 0:
        state["need_more_retrieval"] = True
        missing = unsupported[0]
        state["current_query"] = f"evidence for: {missing[:120]}"
        state["scratchpad"].append("Reflect: found unbacked claims, returning to retrieval")
    else:
        state["need_more_retrieval"] = False
        state["scratchpad"].append("Reflect: claims are backed or budget exhausted")

    return state


def terminate_node(state: AgentState) -> AgentState:
    citations = state["read_chunk_ids"]

    if not citations:
        state["final_answer"] = ""
        state["confidence"] = "low"
        state["status"] = "insufficient_evidence"
        state["refused_reason"] = "No cited evidence could be retrieved within tool-call budget."
        return state

    if state["unsupported_claims"]:
        confidence = "low"
    elif state["open_gaps"]:
        confidence = "medium"
    else:
        confidence = "high"

    drafted = state["draft"].strip()
    if not drafted:
        drafted = "Evidence found, but draft generation did not produce structured claims."

    state["final_answer"] = drafted
    state["confidence"] = confidence
    state["status"] = "ok" if drafted else "insufficient_evidence"
    state["refused_reason"] = "" if confidence != "low" or not state["open_gaps"] else "Some parts remain weakly supported."
    return state


def route_after_sufficiency(state: AgentState) -> str:
    if state["enough_evidence"]:
        return "draft"
    if budget_remaining(state) <= 0:
        return "terminate"
    return "retrieve_read"


def route_after_reflect(state: AgentState) -> str:
    if state["need_more_retrieval"] and budget_remaining(state) > 0:
        return "retrieve_read"
    return "terminate"

In [21]:
# === LANGFUSE TRACING SETUP ===

LANGFUSE_CLIENT = None
if ENABLE_LANGFUSE and Langfuse is not None:
    try:
        LANGFUSE_CLIENT = Langfuse(
            public_key=LANGFUSE_PUBLIC_KEY,
            secret_key=LANGFUSE_SECRET_KEY,
            host=LANGFUSE_HOST,
        )
        print("✓ Langfuse client initialized")
    except Exception as e:
        print(f"✗ Langfuse init failed: {e}")
        LANGFUSE_CLIENT = None


def _safe_update_observation(obj: Any, **kwargs: Any) -> None:
    if obj is None:
        return
    try:
        if hasattr(obj, "update"):
            obj.update(**kwargs)
    except Exception:
        pass


def _safe_end_observation(obj: Any, **kwargs: Any) -> None:
    if obj is None:
        return
    try:
        if hasattr(obj, "end"):
            obj.end(**kwargs)
    except Exception:
        pass


def _create_trace(state: Dict[str, Any], question: str) -> Any:
    if LANGFUSE_CLIENT is None:
        return None
    trace_id = state.get("request_id") or str(uuid.uuid4())
    state["request_id"] = trace_id
    try:
        return LANGFUSE_CLIENT.trace(
            id=trace_id,
            name="langgraph_reflective_rag",
            input={"question": question},
            metadata={
                "pipeline_version": "v1",
                "search_mode": SEARCH_MODE,
            },
        )
    except Exception:
        return None


@contextmanager
def trace_span(state: Dict[str, Any], name: str, input_payload: Optional[Dict[str, Any]] = None):
    trace = state.get("_trace")
    span = None
    if trace is not None:
        try:
            if hasattr(trace, "span"):
                span = trace.span(name=name, input=input_payload)
        except Exception:
            span = None
    yield span


def flush_langfuse() -> None:
    if LANGFUSE_CLIENT is None:
        return
    try:
        if hasattr(LANGFUSE_CLIENT, "flush"):
            LANGFUSE_CLIENT.flush()
    except Exception:
        pass

✓ Langfuse client initialized


In [22]:
# === PRODUCTION WRAPPER NODES ===

def init_state_prod(question: str) -> AgentState:
    state = init_state(question)
    state["request_id"] = str(uuid.uuid4())
    state["warnings"] = []
    state["_trace"] = _create_trace(state, question)
    return state


def search_tool_prod(state: AgentState, query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    with trace_span(state, "search_tool", {"query": query, "top_k": top_k}) as obs:
        t0 = time.perf_counter()
        results = search_tool(query, top_k)
        _safe_end_observation(
            obs,
            output={"result_count": len(results)},
            metadata={"duration_ms": round((time.perf_counter() - t0) * 1000, 2)},
        )
        return results


def read_chunk_tool_prod(state: AgentState, chunk_id: str) -> Dict[str, Any] | None:
    with trace_span(state, "read_chunk_tool", {"chunk_id": chunk_id}) as obs:
        chunk = read_chunk_tool(state, chunk_id)
        _safe_end_observation(obs, output={"found": bool(chunk)})
        return chunk


def generate_with_ollama(prompt: str, context_chunks: List[Dict[str, Any]]) -> str:
    context_lines = []
    for c in context_chunks:
        cid = c.get("chunk_id", "")
        text = (c.get("content") or "").strip().replace("\n", " ")[:1200]
        context_lines.append(f"[{cid}] {text}")

    system_prompt = "You are a careful RAG assistant. Use only provided evidence. If evidence is weak, say so explicitly."
    user_prompt = (
        f"Question:\n{prompt}\n\n"
        f"Evidence chunks:\n" + "\n".join(context_lines) + "\n\n"
        "Return a concise answer with explicit chunk id citations like [doc:chunk]."
    )

    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
    }
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json=payload,
        timeout=OLLAMA_TIMEOUT_SECONDS,
    )
    if resp.status_code == 404:
        payload2 = {"model": OLLAMA_MODEL, "prompt": system_prompt + "\n\n" + user_prompt, "stream": False}
        resp2 = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload2, timeout=OLLAMA_TIMEOUT_SECONDS)
        resp2.raise_for_status()
        return (resp2.json().get("response") or "").strip()

    resp.raise_for_status()
    message = resp.json().get("message", {})
    return (message.get("content") or "").strip()


def plan_node_prod(state: AgentState) -> AgentState:
    with trace_span(state, "plan_node") as obs:
        state = plan_node(state)
        _safe_end_observation(obs, output={"sub_questions": state.get("sub_questions", [])})
        return state


def retrieve_read_node_prod(state: AgentState) -> AgentState:
    with trace_span(state, "retrieve_read_node", {"query": state.get("current_query", "")}) as obs:
        if budget_remaining(state) <= 0:
            state["scratchpad"].append("Budget exhausted before retrieval")
            _safe_end_observation(obs, output={"budget_exhausted": True})
            return state

        query = state["current_query"]
        try:
            results = search_tool_prod(state, query, top_k=TOP_K)
        except Exception as e:
            state["scratchpad"].append(f"Search failed for '{query}': {e}")
            state.setdefault("warnings", []).append(f"Search error: {e}")
            _safe_end_observation(obs, output={"error": str(e)})
            return state

        state["tool_calls_used"] += 1
        state["scratchpad"].append(f"search('{query}') -> {len(results)} results")

        if not results:
            state["open_gaps"] = list(dict.fromkeys(state["open_gaps"] + [query]))
            _safe_end_observation(obs, output={"result_count": 0})
            return state

        to_read: List[str] = []
        for item in results[:2]:
            cid = item.get("chunk_id")
            if cid:
                to_read.append(cid)
                state["retrieved_chunks"][cid] = item

        for cid in to_read:
            if budget_remaining(state) <= 0:
                break
            chunk = read_chunk_tool_prod(state, cid)
            if not chunk:
                continue
            state["tool_calls_used"] += 1
            if cid not in state["read_chunk_ids"]:
                state["read_chunk_ids"].append(cid)

            linked_subq = state["open_gaps"][0] if state["open_gaps"] else state["question"]
            state["evidence_by_question"].setdefault(linked_subq, [])
            if cid not in state["evidence_by_question"][linked_subq]:
                state["evidence_by_question"][linked_subq].append(cid)

            text = (chunk.get("content") or "").strip()
            if text:
                state["known_facts"].append(text[:280])

        if state["open_gaps"]:
            state["current_query"] = state["open_gaps"][0]

        _safe_end_observation(
            obs,
            output={"tool_calls_used": state["tool_calls_used"], "read_chunk_ids": state["read_chunk_ids"]},
        )
        return state


def check_sufficiency_node_prod(state: AgentState) -> AgentState:
    with trace_span(state, "check_sufficiency_node") as obs:
        state = check_sufficiency_node(state)
        _safe_end_observation(
            obs, output={"enough_evidence": state["enough_evidence"], "open_gaps": state.get("open_gaps", [])}
        )
        return state


def draft_node_prod(state: AgentState) -> AgentState:
    with trace_span(state, "draft_node") as obs:
        if not state["read_chunk_ids"]:
            state["draft"] = ""
            _safe_end_observation(obs, output={"drafted": False})
            return state

        context_chunks = [
            state["retrieved_chunks"][cid] for cid in state["read_chunk_ids"] if cid in state["retrieved_chunks"]
        ]

        if ENABLE_OLLAMA_GENERATION:
            try:
                with trace_span(
                    state, "generate_with_ollama", {"model": OLLAMA_MODEL, "chunks": len(context_chunks)}
                ) as gen_obs:
                    text = generate_with_ollama(state["question"], context_chunks)
                    _safe_end_observation(gen_obs, output={"response_chars": len(text)})

                citations = " ".join([f"[{c}]" for c in state["read_chunk_ids"]])
                state["draft"] = text + "\n\nCitations: " + citations
                claims = [ln.strip() for ln in text.split("\n") if ln.strip()]
                state["claims"] = claims
                state["scratchpad"].append("Draft generated with local Ollama")
                _safe_end_observation(obs, output={"draft_mode": "ollama", "claims": len(claims)})
                return state
            except Exception as e:
                state.setdefault("warnings", []).append(f"Ollama generation failed: {e}")
                state["scratchpad"].append(f"Ollama fallback: {e}")

        state = draft_node(state)
        _safe_end_observation(obs, output={"draft_mode": "fallback", "claims": len(state.get("claims", []))})
        return state


def reflect_node_prod(state: AgentState) -> AgentState:
    with trace_span(state, "reflect_node") as obs:
        state = reflect_node(state)
        _safe_end_observation(
            obs,
            output={
                "unsupported_claims": len(state.get("unsupported_claims", [])),
                "need_more_retrieval": state.get("need_more_retrieval", False),
            },
        )
        return state


def terminate_node_prod(state: AgentState) -> AgentState:
    with trace_span(state, "terminate_node") as obs:
        state = terminate_node(state)
        _safe_end_observation(
            obs,
            output={"status": state.get("status"), "confidence": state.get("confidence")},
        )
        trace = state.get("_trace")
        _safe_update_observation(
            trace,
            output={
                "status": state.get("status"),
                "confidence": state.get("confidence"),
                "citations": state.get("read_chunk_ids", []),
            },
            metadata={"tool_calls_used": state.get("tool_calls_used", 0), "warnings": state.get("warnings", [])},
        )
        flush_langfuse()
        return state

In [23]:
# === PRODUCTION GRAPH ===

graph_builder_prod = StateGraph(AgentState)

graph_builder_prod.add_node("plan", plan_node_prod)
graph_builder_prod.add_node("retrieve_read", retrieve_read_node_prod)
graph_builder_prod.add_node("check_sufficiency", check_sufficiency_node_prod)
graph_builder_prod.add_node("draft", draft_node_prod)
graph_builder_prod.add_node("reflect", reflect_node_prod)
graph_builder_prod.add_node("terminate", terminate_node_prod)

graph_builder_prod.set_entry_point("plan")
graph_builder_prod.add_edge("plan", "retrieve_read")
graph_builder_prod.add_edge("retrieve_read", "check_sufficiency")
graph_builder_prod.add_conditional_edges(
    "check_sufficiency",
    route_after_sufficiency,
    {"retrieve_read": "retrieve_read", "draft": "draft", "terminate": "terminate"},
)
graph_builder_prod.add_edge("draft", "reflect")
graph_builder_prod.add_conditional_edges(
    "reflect", route_after_reflect, {"retrieve_read": "retrieve_read", "terminate": "terminate"}
)
graph_builder_prod.add_edge("terminate", END)

graph_prod = graph_builder_prod.compile()
print("✓ Production graph compiled")

✓ Production graph compiled


In [24]:
# === RUNTIME FUNCTIONS ===

def run_agent(question: str) -> Dict[str, Any]:
    state0 = init_state_prod(question)
    result = graph_prod.invoke(state0, config={"recursion_limit": 40})
    return {
        "answer": result.get("final_answer", ""),
        "citations": result.get("read_chunk_ids", []),
        "confidence": result.get("confidence", "low"),
        "status": result.get("status", "insufficient_evidence"),
        "tool_calls_used": result.get("tool_calls_used", 0),
        "warnings": result.get("warnings", []),
        "scratchpad": result.get("scratchpad", []),
        "request_id": result.get("request_id", ""),
    }


def run_agent_markdown(question: str) -> str:
    question = (question or "").strip()
    if not question:
        return "Please enter a question."

    try:
        out = run_agent(question)
    except Exception as e:
        return f"Agent failed: {e}"

    citations = out["citations"] or []
    citations_text = "\n".join([f"- `{c}`" for c in citations]) if citations else "- None"
    warnings = out["warnings"] or []
    warnings_text = "\n".join([f"- {w}" for w in warnings]) if warnings else "- None"
    scratch = out["scratchpad"][-6:]
    scratch_text = "\n".join([f"- {s}" for s in scratch]) if scratch else "- None"

    return (
        f"## Answer\n{out['answer'] or 'No answer generated.'}\n\n"
        f"## Meta\n"
        f"- status: `{out['status']}`\n"
        f"- confidence: `{out['confidence']}`\n"
        f"- tool_calls_used: `{out['tool_calls_used']}`\n"
        f"- request_id: `{out['request_id']}`\n\n"
        f"## Citations\n{citations_text}\n\n"
        f"## Warnings\n{warnings_text}\n\n"
        f"## Scratchpad (tail)\n{scratch_text}"
    )

In [25]:
# === GRADIO INTERFACE ===

gradio_app = gr.Interface(
    fn=run_agent_markdown,
    inputs=gr.Textbox(lines=3, label="Question"),
    outputs=gr.Markdown(label="Agent Output"),
    title="Reflective RAG Agent (Ollama + Langfuse)",
    description="Retrieval uses local /query API (v1); generation uses local Ollama when enabled. Traces sent to Langfuse (if configured).",
    allow_flagging="never",
)

print("✓ Gradio interface created")

✓ Gradio interface created


In [29]:
# Launch UI locally (uncomment to run)
gradio_app.launch(share=False, inline=True)

Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [37]:
# === SMOKE TEST 1: Direct agent invocation ===
smoke_q = "What score did Palm-2L + step back achieve on MMLU physics?"
try:
    smoke_out = run_agent(smoke_q)
    print("✓ Smoke test 1 passed")
    print(f"  Status: {smoke_out['status']}")
    print(f"  Confidence: {smoke_out['confidence']}")
    print(f"  Tool calls: {smoke_out['tool_calls_used']}")
    print(f"  Citations: {len(smoke_out['citations'])}")
except Exception as e:
    print(f"✗ Smoke test 1 failed: {e}")

✓ Smoke test 1 passed
  Status: ok
  Confidence: low
  Tool calls: 8
  Citations: 4


In [36]:
print(smoke_out["request_id"])

f9f4c587-5706-4bf8-8c9e-0dcf56a781c7


In [28]:
# === SMOKE TEST 2: Gradio function path ===
try:
    preview = run_agent_markdown("Summarize the retrieval pipeline.")
    print("✓ Smoke test 2 passed")
    print(f"  Output length: {len(preview)} chars")
    print(f"  Preview: {preview[:200]}...")
except Exception as e:
    print(f"✗ Smoke test 2 failed: {e}")

✓ Smoke test 2 passed
  Output length: 1180 chars
  Preview: ## Answer
- Summarize the retrieval pipeline: this section, we discuss possible future research directions that may further amplify GPT-4V s capabilities. The discussion focuses on how the intriguing ...
